# El Zen de Python, con ejemplos

**Unidad 1 · Semana 1 · Notebook 2 de 2**

Esta notebook se estudia **después** del curso acelerado. El Zen no es un
reglamento ni un examen de memoria: es una colección de criterios para conversar
sobre diseño, legibilidad y mantenimiento.

## Objetivos

- Interpretar las 19 reglas del Zen en contexto.
- Comparar soluciones que funcionan, pero comunican intenciones diferentes.
- Practicar refactorizaciones pequeñas sin cambiar el comportamiento.

## Cómo leer el Zen

Python incluye el texto como un pequeño *easter egg*. Al importar `this`, el Zen
se imprime en inglés (solo la primera vez en cada sesión).

Seguiremos las 19 reglas publicadas por Tim Peters en [PEP 20](https://peps.python.org/pep-0020/).
Las explicaciones están en español; los nombres, comentarios y mensajes del código
están en inglés. Ejecuta las celdas en orden: los ejemplos incluyen sus datos y no
requieren archivos previos ni entrada interactiva.

Las comparaciones ilustran decisiones de diseño, no prohibiciones absolutas.
Una alternativa más larga puede ser apropiada en otro contexto.

## Las 19 reglas

In [ ]:
import this

### 1. Bonito es mejor que feo

La belleza de un programa se aprecia en la claridad de su intención. El formato
consistente y las expresiones fáciles de leer ayudan a reconocer lo que hace.

Ambos saludos son correctos. La cadena formateada permite ver el mensaje completo;
la concatenación también es válida, aunque se vuelve difícil de seguir cuando
hay muchos fragmentos.

In [ ]:
# Beautiful code (follows rule 1): the message is visible as a single template.
def greet(name):
    return f"Hello, {name}!"


# Ugly code (counterexample to rule 1): more fragments obscure the message.
# This is a relative comparison; concatenation itself is valid.
def greet_with_concatenation(name):
    return "Hello, " + name + "!"


print(greet("Alice"))
assert greet("Alice") == greet_with_concatenation("Alice")

In [ ]:
colors = ["red", "green", "blue"]

# Beautiful code (follows rule 1): direct iteration makes the intent clear.
for color in colors:
    print(color)

# Ugly code (counterexample to rule 1): unnecessary indexing adds clutter.
index = 0
while index < len(colors):
    print(colors[index])
    index += 1

### 2. Explícito es mejor que implícito

Haz visibles las dependencias y las decisiones relevantes. El prefijo `math`
permite saber de dónde viene una función. Un nombre como `tax_rate` explica qué
representa un número; no obliga a reconstruir el contexto.

No significa escribir todos los valores por defecto: añade información cuando
ayude a entender la intención.

In [ ]:
# Implicit code (counterexample to rule 2): a wildcard import hides name origins.
from math import *

implicit_result = sqrt(25)

# Explicit code (follows rule 2): the module name makes the dependency visible.
import math

explicit_result = math.sqrt(25)
print(explicit_result)
assert implicit_result == explicit_result

In [ ]:
price = 100

# Implicit code (counterexample to rule 2): the tax rate has no name.
implicit_total = price * 1.16

# Explicit code (follows rule 2): naming the rate explains the calculation.
tax_rate = 0.16
explicit_total = price * (1 + tax_rate)
print(f"Total: {explicit_total:.2f}")
assert math.isclose(implicit_total, explicit_total)

### 3. Simple es mejor que complejo

Si una operación incorporada expresa exactamente lo que necesitas, úsala.
Acumular a mano no está mal, pero aquí añade pasos sin aportar una regla nueva.
Si cada elemento necesitara una validación particular, el bucle podría tener sentido.

In [ ]:
# Simple code (follows rule 3): use the built-in operation for this task.
def sum_numbers(numbers):
    return sum(numbers)


# Complex code (counterexample to rule 3): manual accumulation adds steps here.
def sum_numbers_manually(numbers):
    total = 0
    for number in numbers:
        total += number
    return total


numbers = [4, 8, 3]
print(sum_numbers(numbers))
assert sum_numbers(numbers) == sum_numbers_manually(numbers)

### 4. Complejo es mejor que complicado

Algunos problemas requieren varios pasos. Podemos aceptar esa complejidad y
organizarla, en lugar de comprimirla en una expresión difícil de seguir.

Una compra puede incluir descuentos distintos por producto. Ambas versiones
calculan el mismo total; la segunda da nombre a cada parte de la operación.
Un bucle no es peor que una comprensión: depende de cuánto trabajo contenga.

In [ ]:
cart = [
    {"name": "notebook", "price": 10, "quantity": 2, "discount_rate": 0.10},
    {"name": "pen", "price": 3, "quantity": 4, "discount_rate": 0.0},
]

# Complicated code (counterexample to rule 4): packed operations hide the steps.
compact_total = sum(p["price"] * p["quantity"] * (1 - p["discount_rate"]) for p in cart)

# Complex but clear code (follows rule 4): named steps organize the pricing rules.
def calculate_cart_total(items):
    total = 0
    for item in items:
        subtotal = item["price"] * item["quantity"]
        discount = subtotal * item["discount_rate"]
        total += subtotal - discount
    return total


print(calculate_cart_total(cart))
assert calculate_cart_total(cart) == compact_total

### 5. Plano es mejor que anidado

Las validaciones tempranas reducen niveles de indentación. Este patrón se conoce
como *guard clause*: descartamos pronto los casos que no pueden continuar.

Para editar un documento, la persona debe tener una cuenta activa, el documento
no debe estar archivado y la persona debe ser su propietaria. Las dos funciones
aplican exactamente esas condiciones. Los retornos tempranos permiten leer cada
restricción por separado; no hace falta recordar varios `if` abiertos.

In [ ]:
# Nested code (counterexample to rule 5): conditions add indentation levels.
def can_edit_document_nested(user, document):
    if user["is_active"]:
        if not document["is_archived"]:
            if document["owner_id"] == user["id"]:
                return True
            else:
                return False
        else:
            return False
    else:
        return False


# Flat code (follows rule 5): early returns keep the conditions at one level.
def can_edit_document(user, document):
    if not user["is_active"]:
        return False
    if document["is_archived"]:
        return False
    return document["owner_id"] == user["id"]


user = {"id": 7, "is_active": True}
document = {"owner_id": 7, "is_archived": False}
print(f"Can edit: {can_edit_document(user, document)}")

# Check both versions for every combination of the three conditions.
for is_active in (False, True):
    for is_archived in (False, True):
        for owner_id in (7, 8):
            sample_user = {"id": 7, "is_active": is_active}
            sample_document = {"owner_id": owner_id, "is_archived": is_archived}
            assert can_edit_document(sample_user, sample_document) == (
                can_edit_document_nested(sample_user, sample_document)
            )

### 6. Disperso es mejor que denso

Usa espacios y saltos de línea para separar pasos e ideas. Reducir el número de
líneas no garantiza una lectura más rápida. Tampoco hay que separar todo:
asignaciones como `width, height = 10, 20` pueden ser claras en su contexto.

In [ ]:
# Sparse code (follows rule 6): separate lines and spaces make steps easy to scan.
name = "John"
age = 30
message = f"My name is {name} and I am {age} years old."
print(message)

# Dense code (counterexample to rule 6): operations are squeezed into one line.
name="John"; age=30; message=f"My name is {name} and I am {age} years old."; print(message)

### 7. La legibilidad cuenta

Los nombres describen el dominio; los comentarios deberían explicar decisiones,
no traducir línea por línea lo que el código ya dice.

In [ ]:
# Readable code (follows rule 7): descriptive names communicate meaning and units.
initial_position_meters = 10
speed_meters_per_second = 3
elapsed_seconds = 4
final_position_meters = initial_position_meters + speed_meters_per_second * elapsed_seconds
print(f"Final position: {final_position_meters} meters")

# Less readable code (counterexample to rule 7): names hide meaning and units.
p = 10
v = 3
t = 4
x = p + v * t
assert x == final_position_meters

### 8. Los casos especiales no son lo suficientemente especiales para romper las reglas

Mantén contratos consistentes. Si una función devuelve una lista de nombres,
devolver un texto cuando no encuentra ninguno obliga a tratar ese caso de
otra manera. Una lista vacía ya representa la ausencia de resultados.

Aquí la regla es el contrato de la función, no una prohibición de usar nombres
cortos o de atender situaciones particulares.

In [ ]:
# Inconsistent code (counterexample to rule 8): the empty case breaks the list contract.
def find_active_names_inconsistent(users):
    names = [user["name"] for user in users if user["is_active"]]
    return names if names else "No active users"


print(find_active_names_inconsistent([]))

In [ ]:
# Consistent code (follows rule 8): every case returns a list.
def find_active_names(users):
    return [user["name"] for user in users if user["is_active"]]


users = [
    {"name": "Alice", "is_active": True},
    {"name": "Bob", "is_active": False},
]
print(find_active_names(users))
assert find_active_names([]) == []

### 9. Aunque la practicidad vence a la pureza

La consistencia debe servir a quienes usan el programa. En la presentación de
resultados, un mensaje puede ser más útil que mostrar una lista vacía.
Conservamos el contrato de la función y adaptamos la salida donde corresponde.

Esta regla invita a valorar el costo y el beneficio de una decisión. No exige
construir una arquitectura perfecta para una necesidad pequeña.

In [ ]:
# Practical code (follows rule 9): adapt the display to help the user.
active_names = find_active_names([])
print(", ".join(active_names) if active_names else "No active users")

### 10. Los errores no deberían pasar silenciosamente

Ignorar una excepción puede convertir un defecto visible en datos incorrectos.
Si una cantidad es obligatoria, un texto inválido debe producir un error claro.
Captura solo la excepción que sabes interpretar y conserva su causa.

La celda captura el error al demostrarlo para que la notebook pueda continuar;
la función sigue avisando a quien la llama.

In [ ]:
# Visible errors (follows rule 10): invalid input raises a meaningful exception.
def parse_quantity(text):
    try:
        return int(text)
    except ValueError as error:
        raise ValueError(f"Invalid quantity: {text!r}") from error


print(parse_quantity("3"))
try:
    parse_quantity("three")
except ValueError as error:
    print(f"Expected error: {error}")

In [ ]:
# Silent errors (counterexample to rule 10): invalid input looks like a real zero.
def parse_quantity_silently(text):
    try:
        return int(text)
    except ValueError:
        return 0


print(parse_quantity_silently("three"))
print(parse_quantity_silently("0"))

### 11. A menos que sean silenciados explícitamente

Un error esperado puede tener una respuesta prevista. Si un archivo de
configuración es opcional, su ausencia puede significar «usar los valores por
defecto». Esa decisión debe ser explícita; otros errores, como no tener permiso
para leerlo, deben seguir propagándose.

El ejemplo usa un directorio temporal para no depender de archivos previos ni
modificar archivos del proyecto. Solo silencia `FileNotFoundError`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


# Explicit silencing (follows rule 11): only a missing optional file uses a default.
def read_optional_config(path):
    try:
        return path.read_text(encoding="utf-8")
    except FileNotFoundError:
        # Missing optional configuration means using the documented default.
        return "theme=light"


with TemporaryDirectory() as temporary_directory:
    config_path = Path(temporary_directory) / "config.txt"
    print(read_optional_config(config_path))
    config_path.write_text("theme=dark", encoding="utf-8")
    print(read_optional_config(config_path))

### 12. Ante la ambigüedad, evita la tentación de adivinar

`03/04/2026` puede significar 3 de abril o 4 de marzo. No elijas una interpretación
sin conocer el contrato de entrada: solicita un formato acordado o rechaza la
entrada ambigua. Aquí exigimos `AAAA-MM-DD` y mostramos un error comprensible.

In [ ]:
from datetime import datetime


# Refusing to guess (follows rule 12): enforce the agreed date format.
def parse_event_date(text):
    try:
        parsed_date = datetime.strptime(text, "%Y-%m-%d").date()
    except ValueError as error:
        raise ValueError("Use YYYY-MM-DD, for example 2026-04-03.") from error
    if parsed_date.isoformat() != text:
        raise ValueError("Use YYYY-MM-DD with two-digit months and days.")
    return parsed_date


for date_text in ("2026-04-03", "03/04/2026"):
    try:
        print(parse_event_date(date_text))
    except ValueError as error:
        print(f"Invalid date {date_text!r}: {error}")

### 13. Debería haber una —y preferiblemente solo una— manera obvia de hacerlo

Prefiere las herramientas y convenciones que expresan directamente la tarea.
Para convertir un texto a minúsculas, `str.lower()` comunica la intención sin
hacer que quien lee reconstruya un recorrido carácter por carácter.

No afirma que solo exista una solución válida ni que toda dificultad sea culpa
de la documentación. La forma habitual depende del problema y se aprende con
práctica; una buena API ayuda a reconocerla.

In [ ]:
greeting = "Hello, World!"

# Obvious code (follows rule 13): the string method directly expresses the task.
print(greeting.lower())

# Less obvious code (counterexample to rule 13): an unnecessary loop hides the intent.
print("".join(character.lower() for character in greeting))

### 14. Aunque esa manera puede no ser obvia al principio, a menos que seas neerlandés

El remate humorístico recuerda que lo «obvio» requiere familiaridad con el
lenguaje. No se espera que una persona principiante descubra de inmediato todas
sus convenciones.

Por ejemplo, al aprender Python puedes recorrer índices antes de conocer
`enumerate()`. Leer ejemplos y practicar ayuda a reconocer cuándo una herramienta
expresa mejor la intención. La claridad también depende de lo que conoce tu audiencia.

### 15. Ahora es mejor que nunca

Esperar a tener el diseño perfecto puede impedir que resuelvas un problema real.
Empieza con una solución pequeña que puedas ejecutar, revisar y mejorar.

En un proyecto de análisis, un primer paso útil puede ser cargar un conjunto de
datos y comprobar sus columnas. Ese resultado concreto permite aprender antes
de construir todo el procesamiento. Avanzar no significa omitir validaciones.

### 16. Aunque nunca suele ser mejor que ahora mismo

La regla anterior se equilibra con esta: la prisa no justifica una decisión cuyo
costo todavía no entiendes. A veces conviene posponer una función o descartarla.

Por ejemplo, antes de automatizar el borrado de archivos, define qué se puede
borrar y cómo comprobarás la selección. Mostrar primero los candidatos permite
revisar la decisión. Lo urgente debe evaluarse junto con sus consecuencias.

### 17. Si la implementación es difícil de explicar, es una mala idea

Intenta describir la solución con sus entradas, pasos y resultados. Si para
explicarla necesitas recordar muchas excepciones ocultas, quizá debas dividir
responsabilidades o revisar el diseño.

En el cálculo del carrito de la regla 4, «calcular el subtotal, aplicar el
descuento y acumular» ofrece una explicación verificable. La dificultad propia
del problema no desaparece, pero la implementación no debería añadir confusión innecesaria.

### 18. Si la implementación es fácil de explicar, puede ser una buena idea

La palabra «puede» importa: una explicación sencilla es una señal favorable,
pero no demuestra que el programa sea correcto ni que cubra los requisitos.

«Sumar los valores y dividir entre su cantidad» explica un promedio. Aun así,
hay que decidir qué sucede si no hay valores o si alguno es inválido. Comprueba
los casos límite y el comportamiento esperado además de la claridad del diseño.

### 19. Los espacios de nombres son una gran idea: ¡usemos más de ellos!

Un espacio de nombres agrupa nombres y les da contexto. Los módulos permiten
identificar la procedencia de una operación y evitan mezclar nombres de distintas
bibliotecas. En la regla 2, `math.sqrt()` ya mostraba esa ventaja.

Aquí `math` y `statistics` agrupan operaciones distintas. El prefijo del módulo
ayuda a leer el código; no hace falta inventar clases solo para agrupar funciones.

In [ ]:
# Namespaces (follows rule 19): module prefixes identify where operations belong.
import math
import statistics

measurements = [4, 9, 16]
print(f"Square root: {math.sqrt(measurements[1])}")
print(f"Mean: {statistics.mean(measurements):.2f}")

## El Zen como criterio de diseño

No toda duplicación exige una abstracción y no toda tarea necesita una clase. La
solución apropiada depende del tamaño, la frecuencia de cambio y las personas
que mantendrán el código. El Zen ayuda a formular preguntas; no reemplaza el
juicio profesional.

Preguntas útiles:

- ¿La versión “elegante” se entiende con mayor rapidez?
- ¿Puedo probar cada parte importante de manera aislada?
- ¿La abstracción representa un concepto real o sólo reduce líneas?
- ¿Los errores conservan suficiente contexto para diagnosticarlos?

## Reto de refactorización

La siguiente función mezcla nombres crípticos, anidación y un `except` demasiado
amplio. Reescríbela sin cambiar su contrato: recibe una lista de diccionarios o
`None`, acepta puntuaciones convertibles a números no negativos y devuelve su
promedio, o `None` si no hay ninguna válida. Para este ejercicio, las puntuaciones
son números finitos, textos o `None`.

Intenta resolverlo antes de consultar la solución propuesta en la siguiente
celda. Usa nombres y comentarios en inglés.

In [ ]:
# Counterexample to rules 5, 7, and 10: nesting, cryptic names, and broad silencing.
def f(xs):
    r = []
    for x in xs:
        try:
            if x:
                if "score" in x:
                    if float(x["score"]) >= 0:
                        r.append(float(x["score"]))
        except Exception:
            pass
    return sum(r) / len(r) if r else None


records = [{"score": "0.9"}, None, {}, {"score": "invalid"}, {"score": 0.7}]
print(f(records))

In [ ]:
# Follows rules 5 and 7: early exits and descriptive names clarify the steps.
# Follows rule 11: specific conversion errors are skipped as required by the contract.
def average_valid_scores(records):
    scores = []
    for record in records:
        if not record or "score" not in record:
            continue
        try:
            score = float(record["score"])
        except (TypeError, ValueError):
            continue
        if score >= 0:
            scores.append(score)

    return sum(scores) / len(scores) if scores else None


sample_cases = [
    (records, 0.8),
    ([], None),
    ([None, {}, {"score": "invalid"}, {"score": None}, {"score": -1}], None),
    ([{"score": 0}, {"score": "1"}], 0.5),
]
for sample_records, expected_average in sample_cases:
    assert average_valid_scores(sample_records) == expected_average
    assert average_valid_scores(sample_records) == f(sample_records)

print(average_valid_scores(records))

## Cierre

El código pitónico no es el que utiliza más trucos del lenguaje; es el que hace
evidente su intención, conserva los errores importantes y resulta razonable de
mantener. A lo largo del curso volveremos a estos criterios al diseñar modelos de
datos, pipelines y proyectos reproducibles.